<a href="https://colab.research.google.com/github/asif-malek/superkart_sales_forecast/blob/main/MLOPS_Notebook_superkart_sales_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



<center><font size=6>SuperKart Sales Forecast </center></font>

# Problem Statement

## Business Context

A sales forecast predicts future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action. Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials.
An accurate sales forecast process has many benefits, which include improved decision-making about the future and the reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establishes benchmarks that can be used to assess trends in the future.

## Objective

They hired you as an MLOps Engineer, and your task is to build an automated MLOps pipeline with CI/CD to deliver accurate and reliable sales forecasts. The objective is to leverage historical sales data, industry trends, and the current pipeline status to predict weekly, monthly, quarterly, and annual revenues. By automating data ingestion, preprocessing, model training, evaluation, and deployment, the pipeline will ensure scalability, consistency, and minimal manual intervention. With CI/CD integration, forecasts will be continuously updated and seamlessly deployed, enabling different business verticals to plan sales operations by region, optimize supply chain procurement, reduce risks in sales pipelines, and establish benchmarks for future trend analysis. Ultimately, this solution will enhance decision-making, streamline planning efforts, and drive operational efficiency and business growth.

# Prerequisites

* Create a Github repo
    - Go to ***Github Profile***
    - Click on ***Your repositories*** then select ***New***
      - Repository Name: ***playstore_ad_revenue_prediction***
      - Check the box ***README.md*** file
      - Click on ***Create repository***

* Adding hugging face space secrets to Github Actions to execute the workflow
  1. Go to Hugging Face ***Profile***
  2. Navigate to ***Access Token***
  3. Create a ***New token***
      - Token type ***Write***
      - Token Name ***MLOps***
      - Click on ***Create Token***
      - Copy the generated Token
  4. Now, go to Github repo
      - Click on ***Settings***
      - Navigate to ***Secrets and Variables***
      - Click on ***Actions***
      - Add a ***Repository secerts***
        - Name ***HF_TOKEN***
        - Secret: ***Paste the token created from the hugging face access tokens***
        - Click on ***Add secret***

* Create a Hugging Face space
    - Go to **Hugging Face**
    - Open your **Profile**
    - Click on **New Space**
      - Under the space creation, enter the below details
        - Space name: **play-store-ad-revenue-prediction**
    (If you were trying with different names, be cautious when using a underscore `_` in space names, such as `frontend_space`, as it can cause exceptions when accessing the API URL. Always use an hyphen `-` instead, like `frontend-space`.)
        - Select the space SDK: **Docker**
        - Choose a Docker template: **Streamlit**
        - Click on **Create Space**

In [1]:

import os
os.makedirs("solution", exist_ok=True)

# Model Building

## Data Registration

In [2]:
os.makedirs("solution/data", exist_ok=True)

In [3]:
# Create a folder for storing the model building files
os.makedirs("solution/model_building", exist_ok=True)

In [5]:
%%writefile solution/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
from huggingface_hub import HfApi, create_repo
import os


repo_id = "asifaddicted/superkart-sales-forecast"
repo_type = "dataset"

# Initialize API client
api = HfApi(token=os.getenv("HF_TOKEN"))

# Step 1: Check if the space exists
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Space '{repo_id}' not found. Creating new space...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="solution/data",
    repo_id=repo_id,
    repo_type=repo_type,
)

Writing solution/model_building/data_register.py


## Data Preparation

1. **Imports Necessary Libraries**:

2. **Dataset Loading**:
   - The script defines a path to a dataset stored on Hugging Face and reads it into a Pandas DataFrame.

3. **Data Preparation**:
   - The code creates matrices for predictors (features) and the target variable.
   - It splits the dataset into training and testing sets, reserving 20% of the data for testing. This is done to evaluate the model's performance later.

4. **Saving Prepared Data**:
   - After splitting, the script saves the training and testing datasets (features and target) as CSV files.

5. **Uploading Files**:
   - Finally, it uploads these CSV files back to the Hugging Face Hub, ensuring that they are properly stored in the specified repository.

In [18]:
%%writefile solution/model_building/prep.py
# for data manipulation
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
# for converting text data in to numerical representation
from sklearn.preprocessing import LabelEncoder
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# Define constants for the dataset and output paths
api = HfApi(token=os.getenv("HF_TOKEN"))
DATASET_PATH = "hf://datasets/asifaddicted/superkart-sales-forecast/SuperKart.csv"
df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# Drop unique identifier column (not useful for modeling)
df.drop(columns=['Product_Id', 'Store_Id'], inplace=True)

#This replaces any occurrence of "reg" inside a string, so "reg" → "Regular"
df['Product_Sugar_Content'] = df['Product_Sugar_Content'].str.replace('reg', 'Regular', regex=False)

# Encode categorical columns
label_encoder = LabelEncoder()
df['Product_Sugar_Content'] = label_encoder.fit_transform(df['Product_Sugar_Content'])
df['Product_Type'] = label_encoder.fit_transform(df['Product_Type'])
df['Store_Size'] = label_encoder.fit_transform(df['Store_Size'])
df['Store_Location_City_Type'] = label_encoder.fit_transform(df['Store_Location_City_Type'])
df['Store_Establishment_Year'] = label_encoder.fit_transform(df['Store_Establishment_Year'])

# Define target variable
target_col = 'Product_Store_Sales_Total'

# Split into X (features) and y (target)
X = df.drop(columns=[target_col])
y = df[target_col]

# Perform train-test split
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Xtrain.to_csv("Xtrain.csv",index=False)
Xtest.to_csv("Xtest.csv",index=False)
ytrain.to_csv("ytrain.csv",index=False)
ytest.to_csv("ytest.csv",index=False)


files = ["Xtrain.csv","Xtest.csv","ytrain.csv","ytest.csv"]

for file_path in files:
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_path.split("/")[-1],  # just the filename
        repo_id="asifaddicted/superkart-sales-forecast",
        repo_type="dataset",
    )

Overwriting solution/model_building/prep.py


## Model Training

### Experimentation and Tracking (Development Environment)

In [7]:
!pip install mlflow==3.0.1 pyngrok==7.2.12 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00


To get the ngrok authorization token, please go to this [link](https://dashboard.ngrok.com/authtokens), generate a new token, copy it, and paste it in the designated code line below.


1. **Set Ngrok Authentication**: Authenticates Ngrok using a personal token to enable secure tunneling from local to public network.

2. **Launch MLflow UI**: Starts the MLflow Tracking UI as a background process on local port 5000 for experiment visualization and tracking.

3. **Create Public Tunnel**: Uses Ngrok to expose the local MLflow UI to the internet, generating a public URL that can be accessed remotely.

4. **Display Public URL**: Prints the Ngrok-generated URL, allowing users to open and interact with the MLflow UI in their browser.

In [8]:
from pyngrok import ngrok
import subprocess
import mlflow

# Set your auth token here (replace with your actual token)
ngrok.set_auth_token("3FGHVDSCInb9HBKNGx5jSG1yQWo_6ynA75YsAjGqqMx1S35Mh")

# Start MLflow UI on port 5000
process = subprocess.Popen(["mlflow", "ui", "--port", "5000"])

# Create public tunnel
public_url = ngrok.connect(5000).public_url
print("MLflow UI is available at:", public_url)

MLflow UI is available at: https://guacamole-twelve-limpness.ngrok-free.dev


In [9]:
# Set the tracking URL for MLflow
mlflow.set_tracking_uri(public_url)

# Set the name for the experiment
mlflow.set_experiment("MLOps_CICD_experiment")

2026/06/17 11:09:37 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_CICD_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/197681905662673574', creation_time=1781694577848, experiment_id='197681905662673574', last_update_time=1781694577848, lifecycle_stage='active', name='MLOps_CICD_experiment', tags={}>

In [ ]:
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# for model serialization
import joblib


df = pd.read_csv("solution/data/SuperKart.csv")
print("Dataset loaded successfully.")

# Drop unique identifier column (not useful for modeling)
df.drop(columns=['Product_Id', 'Store_Id'], inplace=True)

# Encode categorical columns
label_encoder = LabelEncoder()


#This replaces any occurrence of "reg" inside a string, so "reg" → "Regular"
df['Product_Sugar_Content'] = df['Product_Sugar_Content'].str.replace('reg', 'Regular', regex=False)



df['Product_Sugar_Content'] = label_encoder.fit_transform(df['Product_Sugar_Content'])
df['Product_Type'] = label_encoder.fit_transform(df['Product_Type'])
df['Store_Size'] = label_encoder.fit_transform(df['Store_Size'])
df['Store_Location_City_Type'] = label_encoder.fit_transform(df['Store_Location_City_Type'])
df['Store_Establishment_Year'] = label_encoder.fit_transform(df['Store_Establishment_Year'])

# Define target variable
target_col = 'Product_Store_Sales_Total'


# Split into X (features) and y (target)
X = df.drop(columns=[target_col])
y = df[target_col]

# Perform train-test split
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Define numeric and categorical features
numeric_features = [
    'Product_Weight', 'Product_Allocated_Area', 'Product_MRP'
]

categorical_features = [
    'Product_Sugar_Content', 'Product_Type', 'Store_Size', 'Store_Establishment_Year', 'Store_Location_City_Type','Store_Type'
]

# Preprocessor
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# Hyperparameter grid
param_grid = {
    'xgbregressor__n_estimators': [50, 100],
    'xgbregressor__max_depth': [3, 5],
    'xgbregressor__learning_rate': [0.01, 0.05],
    'xgbregressor__subsample': [0.7, 0.8],
    'xgbregressor__colsample_bytree': [0.7, 0.8],
    'xgbregressor__reg_lambda': [0.1, 1]
}

# Pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

with mlflow.start_run():
    # Grid Search
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
    grid_search.fit(Xtrain, ytrain)

    # Log parameter sets
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]

        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_neg_mse", mean_score)

    # Best model
    mlflow.log_params(grid_search.best_params_)
    best_model = grid_search.best_estimator_

    # Predictions
    y_pred_train = best_model.predict(Xtrain)
    y_pred_test = best_model.predict(Xtest)

    # Metrics
    train_rmse = mean_squared_error(ytrain, y_pred_train)
    test_rmse = mean_squared_error(ytest, y_pred_test)

    train_mae = mean_absolute_error(ytrain, y_pred_train)
    test_mae = mean_absolute_error(ytest, y_pred_test)

    train_r2 = r2_score(ytrain, y_pred_train)
    test_r2 = r2_score(ytest, y_pred_test)

    # Log metrics
    mlflow.log_metrics({
        "train_RMSE": train_rmse,
        "test_RMSE": test_rmse,
        "train_MAE": train_mae,
        "test_MAE": test_mae,
        "train_R2": train_r2,
        "test_R2": test_r2
    })

Dataset loaded successfully.
🏃 View run sincere-ape-589 at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574/runs/610f02ebc2e447eb81054b14ae1751ff
🧪 View experiment at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574
🏃 View run legendary-chimp-409 at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574/runs/bfc08afc96ce4370aebb0a4e4465db7f
🧪 View experiment at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574
🏃 View run legendary-frog-813 at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574/runs/1195dd25d570484984a8b19606c1aa5b
🧪 View experiment at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574
🏃 View run suave-shoat-564 at: https://guacamole-twelve-limpness.ngrok-free.dev/#/experiments/197681905662673574/runs/63bd915c5aaa4c3a86d9187af8fa9401
🧪 View experiment at: https://guacamole-twelve-l

### Experimentation and Tracking (Production Environment)

In [20]:
%%writefile solution/model_building/train.py
# for data manipulation
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# for model serialization
import joblib
# for creating a folder
import os
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mlops-training-experiment")

api = HfApi()


Xtrain_path = "hf://datasets/asifaddicted/superkart-sales-forecast/play-store-revenue-analysis/Xtrain.csv"
Xtest_path = "hf://datasets/asifaddicted/superkart-sales-forecast/play-store-revenue-analysis/Xtest.csv"
ytrain_path = "hf://datasets/asifaddicted/superkart-sales-forecast/play-store-revenue-analysis/ytrain.csv"
ytest_path = "hf://datasets/asifaddicted/superkart-sales-forecast/ytest.csv"

Xtrain = pd.read_csv(Xtrain_path)
Xtest = pd.read_csv(Xtest_path)
ytrain = pd.read_csv(ytrain_path)
ytest = pd.read_csv(ytest_path)


# Define numeric and categorical features
numeric_features = [
    'Product_Weight', 'Product_Allocated_Area', 'Product_MRP'
]

categorical_features = [
    'Product_Sugar_Content', 'Product_Type', 'Store_Size', 'Store_Establishment_Year', 'Store_Location_City_Type','Store_Type'
]

# Preprocessor
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# Hyperparameter grid
param_grid = {
    'xgbregressor__n_estimators': [50, 100, 150],
    'xgbregressor__max_depth': [3, 5, 7],
    'xgbregressor__learning_rate': [0.01, 0.05, 0.1],
    'xgbregressor__subsample': [0.7, 0.8, 1.0],
    'xgbregressor__colsample_bytree': [0.7, 0.8, 1.0],
    'xgbregressor__reg_lambda': [0.1, 1, 10]
}

# Pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

with mlflow.start_run():
    # Grid Search
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
    grid_search.fit(Xtrain, ytrain)

    # Log parameter sets
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]

        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_neg_mse", mean_score)

    # Best model
    mlflow.log_params(grid_search.best_params_)
    best_model = grid_search.best_estimator_

    # Predictions
    y_pred_train = best_model.predict(Xtrain)
    y_pred_test = best_model.predict(Xtest)

    # Metrics
    train_rmse = mean_squared_error(ytrain, y_pred_train, squared=False)
    test_rmse = mean_squared_error(ytest, y_pred_test, squared=False)

    train_mae = mean_absolute_error(ytrain, y_pred_train)
    test_mae = mean_absolute_error(ytest, y_pred_test)

    train_r2 = r2_score(ytrain, y_pred_train)
    test_r2 = r2_score(ytest, y_pred_test)

    # Log metrics
    mlflow.log_metrics({
        "train_RMSE": train_rmse,
        "test_RMSE": test_rmse,
        "train_MAE": train_mae,
        "test_MAE": test_mae,
        "train_R2": train_r2,
        "test_R2": test_r2
    })

    # Save the model locally
    model_path = "best_sales_forecast_model_v1.joblib"
    joblib.dump(best_model, model_path)

    # Log the model artifact
    mlflow.log_artifact(model_path, artifact_path="model")
    print(f"Model saved as artifact at: {model_path}")

    # Upload to Hugging Face
    repo_id = "asifaddicted/superkart-sales-forecast/sales_forecast_model"
    repo_type = "model"

    # Step 1: Check if the space exists
    try:
        api.repo_info(repo_id=repo_id, repo_type=repo_type)
        print(f"Space '{repo_id}' already exists. Using it.")
    except RepositoryNotFoundError:
        print(f"Space '{repo_id}' not found. Creating new space...")
        create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
        print(f"Space '{repo_id}' created.")

    # create_repo("churn-model", repo_type="model", private=False)
    api.upload_file(
        path_or_fileobj="best_sales_forecast_model_v1.joblib",
        path_in_repo="best_sales_forecast_model_v1.joblib",
        repo_id=repo_id,
        repo_type=repo_type,
    )

Overwriting solution/model_building/train.py


# Deployment

## Dockerfile

In [15]:
os.makedirs("solution/deployment", exist_ok=True)

In [21]:
%%writefile solution/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Overwriting solution/deployment/Dockerfile


## Streamlit App

In [25]:
%%writefile solution/deployment/app.py
import streamlit as st
import pandas as pd
from huggingface_hub import hf_hub_download
import joblib

# Download and load the trained model
model_path = hf_hub_download(repo_id="asifaddicted/superkart-sales-forecast/sales_forecast_model", filename="best_sales_forecast_model_v1.joblib")
model = joblib.load(model_path)

# Streamlit UI
st.title("SuperKart Sales Forecast")
st.write("""
This application predicts the expected **Sales Forecast** of a SuperKart application
based on its characteristics such as Product Sugar Content, installs, active users, and screen time.
Please enter the app details below to get a sales forecast prediction.
""")

# User input
Product_Weight = st.number_input("Product Weight", min_value=4, max_value=22, value=4, step=1)
Product_Sugar_Content = st.selectbox("Product Sugar Content", ["Low Sugar", "Regular", "No Sugar"])
Product_Allocated_Area = st.number_input("Product Allocated Area", min_value=0.001, max_value=10.99, value=0.001, step=0.001)

Product_Type = st.selectbox("Product Type", ["Household", "Starchy Foods", "Dairy", "Snack Foods", "Baking Goods", "Canned", "Frozen Foods", "Hard Drinks", "Meat", "Fruits and Vegetables", "Soft Drinks", "Others", "Health and Hygiene", "Breads", "Seafood", "Breakfast"])
Product_MRP = st.number_input("Product MRP", min_value=0.1, max_value=9999, value=1, step=1)
Store_Establishment_Year = st.number_input("Store Establishment Year", min_value=1900, max_value=2026, value=1900, step=1)
Store_Size = st.selectbox("Store Size", ["High", "Medium", "Small"])
Store_Location_City_Type = st.selectbox("Store City Type", ["Tier 1", "Tier 2", "Tier 3"])
Store_Type = st.selectbox("Store Type", ["Departmental Store", "Supermarket Type1", "Supermarket Type2", "Food Mart"])





# Assemble input into DataFrame
input_data = pd.DataFrame([{
    'Product_Weight': Product_Weight,
    'Product_Sugar_Content': Product_Sugar_Content,
    'Product_Allocated_Area': Product_Allocated_Area,
    'Product_Type': Product_Type,
    'Product_MRP': Product_MRP,
    'Store_Establishment_Year': Store_Establishment_Year,
    'Store_Size': Store_Size,
    'Store_Location_City_Type': Store_Location_City_Type,
    'Store_Type': Store_Type
}])

# Predict button
if st.button("Predict Sales"):
    prediction = model.predict(input_data)[0]
    st.subheader("Prediction Result:")
    st.success(f"Estimated Sales: **${prediction:,.2f} USD**")

Overwriting solution/deployment/app.py


## Dependency Handling

In [23]:
%%writefile solution/deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing solution/deployment/requirements.txt


# Hosting

In [24]:
os.makedirs("solution/hosting", exist_ok=True)

In [26]:
%%writefile solution/hosting/hosting.py
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
api.upload_folder(
    folder_path="solution/deployment",     # the local folder containing your files
    repo_id="asifaddicted/superkart-sales-forecast",          # the target repo
    repo_type="space",                      # dataset, model, or space
    path_in_repo="",                          # optional: subfolder path inside the repo
)

Writing solution/hosting/hosting.py


# Create and Automate MLOps Pipeline with GitHub Action Workflows using CI/CD

## Action Workflow YAML File

* A YAML file is a simple, human-readable file used to store configuration settings.
* YAML stands for Yet Another Markup Language or YAML Ain't Markup Language (a recursive acronym).
* It uses indentation (spaces) to show structure, like folders inside folders.
* Each line contains a key and a value, making it easy to organize data.
* YAML is often used in automation tools, cloud setups, and app settings.

Here's the YAML file we'd need for our use case.

```
name: Solution

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r solution/requirements.txt
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python solution/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r solution/requirements.txt
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python solution/model_building/prep.py


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r solution/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python solution/model_building/train.py


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r solution/requirements.txt
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python solution/hosting/hosting.py

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [27]:
%%writefile solution/requirements.txt
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing solution/requirements.txt


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<--------Gmail Address------>"
!git config --global user.name "<----Github Username-------->"

# Clone your GitHub repository
!git clone https://github.com/<----Github Username-------->/playstore_ad_revenue_prediction.git

# Move your folder to the repository directory
!mv /content/week_3_practice/ /content/playstore_ad_revenue_prediction

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.15).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
Cloning into 'playstore_ad_revenue_prediction'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [ ]:
# Change directory to the cloned repository
%cd playstore_ad_revenue_prediction/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<----Github Username-------->:<----Github Token-------->@github.com/<----Github Username-------->/playstore_ad_revenue_prediction.git

/content/playstore_ad_revenue_prediction
[main f0257ac] first commit
 9 files changed, 94169 insertions(+)
 create mode 100644 week_3_practice/data/playstore_revenue_analysis.csv
 create mode 100644 week_3_practice/deployment/Dockerfile
 create mode 100644 week_3_practice/deployment/app.py
 create mode 100644 week_3_practice/deployment/requirements.txt
 create mode 100644 week_3_practice/hosting/hosting.py
 create mode 100644 week_3_practice/model_building/data_register.py
 create mode 100644 week_3_practice/model_building/prep.py
 create mode 100644 week_3_practice/model_building/train.py
 create mode 100644 week_3_practice/requirements.txt
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (15/15), done.
Writing objects: 100% (16/16), 3.00 MiB | 3.48 MiB/s, done.
Total 16 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/praneeth300/playstore_ad_revenue_prediction.git
   af7801f..f025

<font size=6 color="navyblue">Power Ahead!</font>
___